In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.layers import Input, Dense, Embedding, LSTM
from tensorflow.keras.models import Model

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/project/data.csv')

print(df.head())
print(df.shape)

df = df.dropna(subset=['review', 'sentiment']).copy()

df['sentiment'] = df['sentiment'].str.lower().str.strip()

df['sentiment'] = df['sentiment'].map({
    'positive': 2,
    'neutral': 1,
    'negative': 0
})

df = df.dropna(subset=['sentiment']).copy()

df['sentiment'] = df['sentiment'].astype(int)

x = df['review']
y = df['sentiment']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

                                              review             model  \
0  The movie was a complete snoozefest.  I regret...  Gemini-1.5-Flash   
1  Seriously, the worst venue I've ever been to. ...  Gemini-1.5-Flash   
2  This event was so poorly organized, it was a j...  Gemini-1.5-Flash   
3  What a disaster of a movie!  The acting was aw...  Gemini-1.5-Flash   
4  I wouldn't recommend this venue to my worst en...  Gemini-1.5-Flash   

  sentiment  
0  negative  
1  negative  
2  negative  
3  negative  
4  negative  
(10147, 3)


In [ ]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)

tokenizer.fit_on_texts(x_train)

x_train_seq = tokenizer.texts_to_sequences(x_train)
x_test_seq = tokenizer.texts_to_sequences(x_test)

max_length = 200

x_train_pad = pad_sequences(
    x_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Training shape:", x_train_pad.shape)

x_test_pad = pad_sequences(
    x_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Testing shape:", x_test_pad.shape)

Training shape: (8116, 200)
Testing shape: (2029, 200)


In [ ]:
inputs = Input(shape=(max_length,), name="input_Layer")

embedding = Embedding(
    input_dim=vocab_size,
    output_dim=64,
    mask_zero=True,
    name="Embedding_Layer"
)(inputs)

lstm = LSTM(
    units=64,
    activation='tanh',
    name="LSTM_Layer"
)(embedding)

outputs = Dense(
    units=3,
    activation='softmax',
    name="Dense_Layer"
)(lstm)

model = Model(
    inputs=inputs,
    outputs=outputs,
    name="LSTM_Multiclass_Model"
)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "LSTM_Multiclass_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_Layer         │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Embedding_Layer     │ (None, 200, 64)   │    640,000 │ input_Layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 200)       │          0 │ input_Layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ LSTM_Layer (LSTM)   │ (None, 64)        │     33,024 │ Embedding_Layer[… │
│                     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Dense_Layer (Dense) │ (None, 3)         │        195 │ LSTM_Layer[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 673,219 (2.57 MB)

 Trainable params: 673,219 (2.57 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2
)

Epoch 1/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 31s 533ms/step - accuracy: 0.6610 - loss: 0.8148 - val_accuracy: 0.8467 - val_loss: 0.4257
Epoch 2/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 34s 412ms/step - accuracy: 0.9241 - loss: 0.2512 - val_accuracy: 0.9218 - val_loss: 0.2497
Epoch 3/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 42s 429ms/step - accuracy: 0.9717 - loss: 0.1132 - val_accuracy: 0.9249 - val_loss: 0.2275
Epoch 4/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 41s 432ms/step - accuracy: 0.9900 - loss: 0.0524 - val_accuracy: 0.9200 - val_loss: 0.2684
Epoch 5/5
51/51 ━━━━━━━━━━━━━━━━━━━━ 23s 452ms/step - accuracy: 0.9948 - loss: 0.0318 - val_accuracy: 0.9083 - val_loss: 0.2943


In [ ]:
# Evaluate the model by disabling the mask check if needed, or by using a batch size that avoids the cuDNN error
loss, accuracy = model.evaluate(x_test_pad, y_test, batch_size=128, verbose=1)

print("Test Loss:", loss)
print("Test Accuracy:", round(accuracy, 4))

16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.9138 - loss: 0.2834
Test Loss: 0.28336697816848755
Test Accuracy: 0.9138


In [ ]:
reviews = [
    "I am extremely satisfied with this purchase, highly recommend!",
    "The product is just okay not too bad or good",
    "Not the product I want it was bad"
]

reviewseq = tokenizer.texts_to_sequences(reviews)

reviewpad = pad_sequences(
    reviewseq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

predictions = model.predict(reviewpad)

sentiment_map = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}

predicted_sentiments = []

for p in predictions:
    predicted_class = np.argmax(p)
    predicted_sentiments.append(sentiment_map[predicted_class])

print("Raw Predictions (probabilities for each class [negative, neutral, positive]):\n", predictions)

print("Predicted Sentiments:")

for i, review in enumerate(reviews):
    print(f"Review: '{review}' -> Sentiment: {predicted_sentiments[i]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step
Raw Predictions (probabilities for each class [negative, neutral, positive]):
 [[8.7258348e-05 4.9739316e-05 9.9986303e-01]
 [5.6311104e-02 9.4331127e-01 3.7756638e-04]
 [8.7070990e-01 1.2463993e-01 4.6501625e-03]]
Predicted Sentiments:
Review: 'I am extremely satisfied with this purchase, highly recommend!' -> Sentiment: positive
Review: 'The product is just okay not too bad or good' -> Sentiment: neutral
Review: 'Not the product I want it was bad' -> Sentiment: negative
